In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iabhishekofficial/mobile-price-classification")

print("Path to dataset files:", path)

C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 70.6k/70.6k [00:00<00:00, 399kB/s]

Extracting files...
Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\iabhishekofficial\mobile-price-classification\versions\1


In [1]:
import pandas as pd 
import numpy as np 

train_ds = pd.read_csv("C:\\Users\\USER\\.cache\\kagglehub\\datasets\\iabhishekofficial\\mobile-price-classification\\versions\dataset\\train.csv")
test_ds = pd.read_csv("C:\\Users\\USER\\.cache\\kagglehub\\datasets\\iabhishekofficial\\mobile-price-classification\\versions\dataset\\test.csv")

In [2]:
train_ds.isnull().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [3]:
test_ds.isnull().sum()

id               0
battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
dtype: int64

In [4]:
train_ds["price_range"].unique()

array([1, 2, 3, 0], dtype=int64)

In [12]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================
# Load data
# ==========================

# Replace with your target column name
TARGET = "price_range"

# ==========================
# Separate features and labels
# ==========================
X = train_ds.drop(columns=[TARGET])
y = train_ds[TARGET]

# Remove ID column from features if present
if "id" in X.columns:
    X = X.drop(columns=["id"])

# ==========================
# Train / Validation / Test split (3-way, all from train_ds)
# ==========================
# Step 1: carve off 30% for val+test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Step 2: split that 30% in half -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
# Result: ~70% train, ~15% val (used during training), ~15% test (final unbiased check)

# ==========================
# Scale Features
# ==========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# ==========================
# Build the Neural Network
# ==========================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================
# Train
# ==========================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32
)

# ==========================
# Final, unbiased evaluation
# ==========================
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Held-out Test Accuracy:", test_accuracy)

Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2643 - loss: 1.3873 - val_accuracy: 0.2733 - val_loss: 1.2880
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4436 - loss: 1.1850 - val_accuracy: 0.5333 - val_loss: 1.0962
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5764 - loss: 0.9889 - val_accuracy: 0.5433 - val_loss: 0.9130
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6536 - loss: 0.7875 - val_accuracy: 0.6800 - val_loss: 0.7216
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8079 - loss: 0.5767 - val_accuracy: 0.8200 - val_loss: 0.5301
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8786 - loss: 0.4050 - val_accuracy: 0.8500 - val_loss: 0.4150
Epoch 7/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9193 - loss: 0.2981 - val_accuracy: 0.8900 - val_loss: 0.3299
Epoch 8/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9400 - loss: 0.2292 - val_accuracy: 0.8833 - val_loss:

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test).argmax(axis=1)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
[[69  6  0  0]
 [ 1 70  4  0]
 [ 0  1 70  4]
 [ 0  0  1 74]]
              precision    recall  f1-score   support

           0       0.99      0.92      0.95        75
           1       0.91      0.93      0.92        75
           2       0.93      0.93      0.93        75
           3       0.95      0.99      0.97        75

    accuracy                           0.94       300
   macro avg       0.94      0.94      0.94       300
weighted avg       0.94      0.94      0.94       300



In [14]:
import joblib

# ==========================
# Save the trained model (architecture + weights + optimizer state)
# ==========================
model.save("price_classifier.keras")

# ==========================
# Save the scaler too — you MUST reuse the exact same scaler
# for any future predictions, or the inputs won't match what
# the model was trained on
# ==========================
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved.")

Model and scaler saved.
